In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as patches

# Estilo visual
plt.style.use('dark_background')

# Cargar dataset
df = pd.read_csv('acb_master_playbyplay.csv')

# Quedarnos solo con los tiros con coordenadas válidas
df_shots = df[(df['shot_type'].isin(['2PT', '3PT'])) & 
              (df['posX'].notna()) & 
              (df['posY'].notna()) & 
              ((df['posX'] != 0) | (df['posY'] != 0))].copy()

print(f"Tiros a analizar: {df_shots.shape[0]}")

In [ ]:
# Invertimos los ejes para proyectar el tracking original en una única media cancha vertical
df_shots['posX_norm'] = df_shots['posY'] 
df_shots['posY_norm'] = df_shots['posX'] 

print("Coordenadas normalizadas a media pista.")

In [ ]:
def draw_fiba_half_court(ax=None, color='white', lw=1.5):
    """ Dibuja una media cancha de baloncesto estilo FIBA sobre un eje """
    if ax is None:
        ax = plt.gca()

    # El aro en (0, 0)
    hoop = patches.Circle((0, 0), radius=225, linewidth=lw, color=color, fill=False)

    # Tablero (-900 a 900 en X, -400 en Y aprox)
    backboard = patches.Rectangle((-900, -400), 1800, 10, linewidth=lw, color=color)

    # Pintura (4900 ancho x 5800 largo, desde la línea de fondo Y=-1575 hasta 4225)
    paint = patches.Rectangle((-2450, -1575), 4900, 5800, linewidth=lw, color=color, fill=False)

    # Círculos de tiro libre (Radio 1800) en Y = 4225
    ft_arc_top = patches.Arc((0, 4225), 3600, 3600, theta1=0, theta2=180, linewidth=lw, color=color)
    ft_arc_bottom = patches.Arc((0, 4225), 3600, 3600, theta1=180, theta2=0, linewidth=lw, color=color, linestyle='dashed')

    # Línea de 3 puntos (Arco de 6750)
    # Empieza en los lados a 900mm de la línea de fondo (Y=-675)
    three_pt_arc = patches.Arc((0, 0), 13500, 13500, theta1=22, theta2=158, linewidth=lw, color=color)
    three_pt_left = patches.Rectangle((-6750, -1575), 0, 2550, linewidth=lw, color=color)
    three_pt_right = patches.Rectangle((6750, -1575), 0, 2550, linewidth=lw, color=color)

    # Añadir parches al gráfico
    for element in [hoop, backboard, paint, ft_arc_top, ft_arc_bottom, three_pt_arc, three_pt_left, three_pt_right]:
        ax.add_patch(element)

    return ax

In [ ]:
# CREACIÓN DEL GRÁFICO
fig, ax = plt.subplots(figsize=(10, 9))

# Dibujar la cancha
draw_fiba_half_court(ax, color='white', lw=1.5)

# Añadir el mapa de densidad con las coordenadas normalizadas
sns.kdeplot(
    x=df_shots['posX_norm'], 
    y=df_shots['posY_norm'], 
    cmap="inferno", 
    fill=True, 
    alpha=0.8,
    bw_adjust=0.5, 
    thresh=0.05,       
    levels=15,
    ax=ax
)

# Ajustar los límites (Pista de 15m ancho [-7500 a 7500] y media pista larga 14m [-2000 a 12000])
ax.set_xlim(-8000, 8000)
ax.set_ylim(-2000, 12000)

ax.axis('off')
plt.title('Mapa Topográfico de Tiros - Liga ACB', color='white', fontsize=20, pad=15)
plt.show()